# resale-retrieval-bench — Colab runner

Zero-shot vision-language embeddings on the DeepFashion **Consumer-to-Shop** benchmark.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough). Expect 2–4 hours for all seven models; embeddings and results are written to your Google Drive so a disconnect only costs the model that was running.

The dataset is released for non-commercial research use (Liu et al., CVPR 2016). Keep it in your own Drive; do not redistribute.

In [ ]:
!nvidia-smi -L || echo 'No GPU: switch runtime type to GPU before continuing'

## 1. Mount Drive, unpack the code, install
The zip lives in `My Drive/rrb/`. Outputs go to `My Drive/rrb/out/` so a disconnect loses nothing.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
RRB = "/content/drive/MyDrive/rrb"                 # folder that holds resale-retrieval-bench.zip and this notebook
OUT = f"{RRB}/out"                                 # embeddings, results and figures persist here
os.makedirs(OUT, exist_ok=True)
assert os.path.exists(f"{RRB}/resale-retrieval-bench.zip"), f"resale-retrieval-bench.zip not found in {RRB}"
!cd /content && rm -rf resale-retrieval-bench && unzip -q -o "{RRB}/resale-retrieval-bench.zip"
%cd /content/resale-retrieval-bench
!pip install -q -r requirements.txt
!python -c "import torch, open_clip, transformers; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"
print("outputs ->", OUT)

## 2. Dataset

The Consumer-to-Shop images are **encrypted**. You get the password by emailing the signed DeepFashion Release Agreement (see `RUN_GUIDE.md`, step 1). Nothing below runs without it.

**Option B (use this):** the authors' folder is already a shortcut in your My Drive. Paste the password into `PASSWORD` and run the cell. It fetches the annotation files, copies the image zip to the Colab disk, and unzips it.

**Option A** (gdown download of the whole folder) is kept as a fallback if the shortcut disappears; it also needs the password.

In [ ]:
# Option A (fallback): download the whole folder with gdown, then unzip with the password
PASSWORD = ""          # <- required
DATA = "/content/data"; ROOT = f"{DATA}/Consumer_to_shop"
assert PASSWORD, "Set PASSWORD first."
!bash scripts/download_deepfashion.sh {DATA} "{PASSWORD}"

In [ ]:
# Option B (reliable). The image zip is ENCRYPTED: paste the password the DeepFashion team sends you after the signed agreement.
PASSWORD = ""          # <- required
SHORTCUT = "/content/drive/MyDrive/Consumer-to-shop Clothes Retrieval Benchmark"
DATA = "/content/data"; ROOT = f"{DATA}/Consumer_to_shop"
assert PASSWORD, "Set PASSWORD first. See RUN_GUIDE.md step 1: the password comes from the DeepFashion team."
import glob, shutil, subprocess
os.makedirs(ROOT, exist_ok=True)

# 1) Annotation files. The Drive mount sometimes lists these shared folders as empty; fall back to gdown by folder id.
ANNO_URL = "https://drive.google.com/drive/folders/0B7EVK8r0v71pMVBqM1Z1b29Tcnc?resourcekey=0-NWldFxSChFuCpK4nzAIGsg"
EVAL_URL = "https://drive.google.com/drive/folders/0B7EVK8r0v71pckZOSm42TFVKNW8?resourcekey=0-NWldFxSChFuCpK4nzAIGsg"
for name, url in (("Anno", ANNO_URL), ("Eval", EVAL_URL)):
    dst = os.path.join(ROOT, name)
    src = os.path.join(SHORTCUT, name)
    if os.path.isdir(src) and os.listdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    if not (os.path.isdir(dst) and any(f.endswith(".txt") for f in os.listdir(dst))):
        !pip install -q gdown
        !gdown --folder "{url}" -O "{dst}" --remaining-ok
    print(name, "->", sorted(os.listdir(dst)) if os.path.isdir(dst) else "MISSING")

# 2) Images: copy the zip to local disk first (unzipping straight from the mount fails), then unzip with the password.
if not os.path.exists(f"{DATA}/img.zip"):
    !cp "{SHORTCUT}/Img/img.zip" "{DATA}/img.zip"
!cd "{ROOT}" && unzip -q -o -P "{PASSWORD}" "{DATA}/img.zip"; echo "unzip exit: $?"
!find "{ROOT}/img" -name "*.jpg" | wc -l
print(os.listdir(ROOT))

### Check the parse before spending GPU time

In [ ]:
!python -c "from rrb import data as D; r=D.build_records('{ROOT}'); print(D.summarize(r)); q,g=D.select_sets(r); print(len(q),'queries', len(g),'gallery')" 

## 3. Embed (one model per call; re-running skips finished models)

In [ ]:
MODELS = ["clip-vit-b32", "clip-vit-l14", "siglip-b16", "fashionclip", "marqo-fashionclip", "marqo-fashionsiglip", "dinov2-b"]
for m in MODELS:
    !python -m rrb.embed --root {ROOT} --out {OUT} --model {m} --crop bbox --batch-size 256 --workers 4
# Crop ablation on two models
for m in ["clip-vit-b32", "marqo-fashionsiglip"]:
    !python -m rrb.embed --root {ROOT} --out {OUT} --model {m} --crop none --batch-size 256 --workers 4

## 4. Evaluate and report

In [ ]:
!python -m rrb.evaluate --out {OUT}
!python -m rrb.report --out {OUT}
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{OUT}/report/results.md").read()))
for f in ["fig1_c2s_recall.png", "fig2_gap_dumbbell.png", "fig3_type_heatmap.png"]:
    display(Image(filename=f"{OUT}/report/{f}", width=900))